# GLOBE Notebook 6 — Interface Web ISIC
**ISIC Project | Segmentation de Lésions Cutanées**

Ce notebook lance une **interface web interactive** avec Gradio directement dans Colab.

### Fonctionnalités de l'interface :
- OUTBOX_TRAY **Uploader** une image dermoscopique
- STUDIO_MICROPHONE **Ajuster** le seuil de segmentation en temps réel
- EYE **Visualiser** : image originale, carte de probabilité, masque, overlay
- REPORT **Afficher** les métriques si un masque réel est fourni
- SAVE **Télécharger** le masque prédit

> **Pré-requis** : avoir exécuté `03_Modele_et_Entrainement.ipynb` pour générer `best_unet_isic.pth`

## SETTING Étape 1 — Setup & Connexion Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ISIC_Project'
MODEL_PATH   = os.path.join(PROJECT_PATH, 'outputs', 'best_unet_isic.pth')
OUTPUTS_PATH = os.path.join(PROJECT_PATH, 'outputs')

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
    print(f'OK Modèle trouvé ({size_mb:.1f} MB)')
else:
    print('CROSS_MARK Modèle introuvable — lance d\'abord le notebook 03 !')

MessageError: Error: credential propagation was unsuccessful

## PACKAGE Étape 2 — Installation des dépendances

In [ ]:
!pip install -q gradio albumentations opencv-python-headless
print('OK Dépendances installées !')

## BRAIN Étape 3 — Chargement du modèle

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

# ── U-Net ────────────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)
        for f in features:
            self.downs.append(DoubleConv(in_channels, f))
            in_channels = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = self.ups[i+1](torch.cat([skip, x], dim=1))
        return self.final(x)

model = UNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'OK Modèle U-Net chargé et prêt !')

# ── Transform ────────────────────────────────────────────────────────────────
IMG_SIZE = 256
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

## TOOL Étape 4 — Fonctions de prédiction et de visualisation

In [ ]:
def predict(image_np, threshold=0.5):
    """Prédit le masque à partir d'un tableau numpy RGB."""
    aug = transform(image=image_np)
    tensor = aug['image'].unsqueeze(0).float().to(device)
    with torch.no_grad():
        logit    = model(tensor)
        prob_map = torch.sigmoid(logit).squeeze().cpu().numpy()
        mask     = (prob_map > threshold).astype(np.uint8)
    # Retaille masque à la résolution originale
    h, w = image_np.shape[:2]
    mask_orig = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return prob_map, mask, mask_orig


def compute_metrics(pred, true, smooth=1e-6):
    p = pred.flatten().astype(float)
    t = true.flatten().astype(float)
    tp = (p * t).sum()
    fp = (p * (1-t)).sum()
    fn = ((1-p) * t).sum()
    return {
        'Dice'      : (2*tp+smooth)/(2*tp+fp+fn+smooth),
        'IoU'       : (tp+smooth)/(tp+fp+fn+smooth),
        'Précision' : (tp+smooth)/(tp+fp+smooth),
        'Recall'    : (tp+smooth)/(tp+fn+smooth),
    }


def make_colormap(prob_map):
    """Carte de probabilité colorée (jet) → PIL Image."""
    cm = plt.cm.get_cmap('jet')
    colored = (cm(prob_map)[:,:,:3] * 255).astype(np.uint8)
    return Image.fromarray(colored)


def make_overlay(image_np, mask, alpha=0.45):
    """Superpose le masque (rouge) sur l'image → PIL Image."""
    img_r = cv2.resize(image_np, (IMG_SIZE, IMG_SIZE))
    overlay = img_r.copy()
    overlay[mask > 0] = [255, 60, 60]
    blended = cv2.addWeighted(img_r, 1-alpha, overlay, alpha, 0)
    # Contour de la lésion
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(blended, contours, -1, (255, 220, 0), 2)
    return Image.fromarray(blended)


def make_mask_visual(mask):
    """Masque binaire en image PIL (blanc=lésion, noir=sain)."""
    return Image.fromarray((mask * 255).astype(np.uint8))


def make_stats_figure(prob_map, mask, metrics=None):
    """Génère un graphique des statistiques → PIL Image."""
    fig, axes = plt.subplots(1, 2 if metrics else 1, figsize=(10, 3.5))
    if metrics is None:
        axes = [axes]

    # Distribution des probabilités
    axes[0].hist(prob_map.flatten(), bins=50,
                 color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(0.5, color='red', lw=2, linestyle='--', label='Seuil 0.5')
    axes[0].set_title('Distribution des probabilités')
    axes[0].set_xlabel('Probabilité'); axes[0].set_ylabel('Fréquence')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Métriques si disponibles
    if metrics:
        names  = list(metrics.keys())
        values = list(metrics.values())
        colors = ['#0E7C7B','#14B8A6','#F26419','#334155']
        bars   = axes[1].bar(names, values, color=colors, edgecolor='white')
        for bar, val in zip(bars, values):
            axes[1].text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
        axes[1].set_ylim(0, 1.15)
        axes[1].set_title('Métriques vs masque réel')
        axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    return Image.open(buf)


print('OK Fonctions prêtes !')

## GLOBE Étape 5 — Lancement de l'interface Gradio

In [ ]:
import gradio as gr

# ── Fonction principale appelée par l'interface ───────────────────────────────
def segment_image(image, mask_reel, threshold, show_contour):
    """
    Fonction principale de l'interface.
    Args:
        image       : image uploadée (PIL)
        mask_reel   : masque réel optionnel (PIL)
        threshold   : seuil de décision (float)
        show_contour: afficher le contour jaune (bool)
    Returns:
        Tuple de (PIL images, texte métriques)
    """
    if image is None:
        return None, None, None, None, 'WARNING Veuillez uploader une image.'

    # Convertir PIL → numpy RGB
    img_np = np.array(image.convert('RGB'))

    # Prédiction
    prob_map, mask_256, mask_orig = predict(img_np, threshold=threshold)

    # Surface lésion
    lesion_pct = mask_256.sum() / mask_256.size * 100

    # Overlay
    alpha  = 0.45
    img_r  = cv2.resize(img_np, (IMG_SIZE, IMG_SIZE))
    overlay = img_r.copy()
    overlay[mask_256 > 0] = [255, 60, 60]
    blended = cv2.addWeighted(img_r, 1-alpha, overlay, alpha, 0)
    if show_contour:
        contours, _ = cv2.findContours(
            mask_256.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(blended, contours, -1, (255, 220, 0), 2)
    overlay_pil = Image.fromarray(blended)

    # Carte de probabilité
    cm       = plt.cm.get_cmap('jet')
    prob_rgb = (cm(prob_map)[:,:,:3] * 255).astype(np.uint8)
    prob_pil = Image.fromarray(prob_rgb)

    # Masque binaire
    mask_pil = Image.fromarray((mask_256 * 255).astype(np.uint8))

    # Métriques si masque réel fourni
    metrics = None
    if mask_reel is not None:
        mask_true = np.array(mask_reel.convert('L'))
        mask_true = cv2.resize((mask_true > 127).astype(np.uint8),
                               (IMG_SIZE, IMG_SIZE),
                               interpolation=cv2.INTER_NEAREST)
        metrics = compute_metrics(mask_256, mask_true)

    # Graphique stats
    stats_pil = make_stats_figure(prob_map, mask_256, metrics)

    # Texte résumé
    lines = [
        f'TRIANGLE_RULER Taille image        : {img_np.shape[1]}×{img_np.shape[0]} px',
        f'TARGET Seuil utilisé       : {threshold:.2f}',
        f'RED_CIRCLE Surface lésion      : {lesion_pct:.1f}% de l\'image',
        f'🔢 Pixels lésion       : {mask_256.sum():,} / {mask_256.size:,}',
    ]
    if metrics:
        lines += [
            '',
            'REPORT Métriques (vs masque réel) :',
            f'   Dice Score  : {metrics["Dice"]:.4f}',
            f'   IoU         : {metrics["IoU"]:.4f}',
            f'   Précision   : {metrics["Précision"]:.4f}',
            f'   Recall      : {metrics["Recall"]:.4f}',
        ]
        if metrics['Recall'] < 0.7:
            lines.append('WARNING  Recall faible → Beaucoup de lésions manquées')
        if metrics['Dice'] > 0.85:
            lines.append('OK  Excellente segmentation !')
        elif metrics['Dice'] > 0.70:
            lines.append('YELLOW_CIRCLE  Bonne segmentation')
        else:
            lines.append('RED_CIRCLE  Segmentation difficile — essaie d\'ajuster le seuil')
    else:
        lines.append('')
        lines.append('BULB Upload un masque réel pour voir les métriques')

    return overlay_pil, prob_pil, mask_pil, stats_pil, '\n'.join(lines)


# ── Construction de l'interface ───────────────────────────────────────────────
with gr.Blocks(
    title='ISIC Segmentation — Interface',
    theme=gr.themes.Soft(primary_hue='teal', secondary_hue='orange'),
    css="""
        .gradio-container { font-family: Arial, sans-serif; }
        .title-box { text-align: center; padding: 10px; }
        footer { display: none !important; }
    """
) as demo:

    # ── En-tête ──────────────────────────────────────────────────────────────
    gr.HTML("""
    <div style="background: linear-gradient(135deg, #0D1B2A, #0E7C7B);
                padding: 24px; border-radius: 12px; margin-bottom: 16px; text-align: center;">
        <h1 style="color: white; margin: 0; font-size: 2em;">MICROSCOPE ISIC Segmentation</h1>
        <p style="color: #CCFBF1; margin: 6px 0 0 0; font-size: 1.1em;">
            Détection automatique de lésions cutanées — U-Net · PyTorch
        </p>
    </div>
    """)

    # ── Corps principal ───────────────────────────────────────────────────────
    with gr.Row():

        # Colonne gauche : inputs
        with gr.Column(scale=1):
            gr.Markdown("### OUTBOX_TRAY Entrées")

            input_image = gr.Image(
                label="Image dermoscopique",
                type="pil",
                image_mode="RGB",
                height=260,
            )
            mask_input = gr.Image(
                label="Masque réel (optionnel — pour les métriques)",
                type="pil",
                image_mode="L",
                height=200,
            )

            gr.Markdown("### STUDIO_MICROPHONE Paramètres")
            threshold_slider = gr.Slider(
                minimum=0.1, maximum=0.9, value=0.5, step=0.05,
                label="Seuil de décision",
                info="0.3 = plus sensible (moins de manqués) · 0.7 = plus strict (moins de fausses alarmes)"
            )
            contour_check = gr.Checkbox(
                value=True,
                label="Afficher le contour de la lésion (jaune)"
            )

            btn = gr.Button(
                "MAGNIFYING_GLASS Segmenter",
                variant="primary",
                size="lg"
            )

        # Colonne droite : outputs
        with gr.Column(scale=2):
            gr.Markdown("### REPORT Résultats")

            with gr.Row():
                out_overlay = gr.Image(
                    label="PALETTE Overlay (lésion en rouge)",
                    height=220,
                )
                out_prob = gr.Image(
                    label="THERMOMETER Carte de probabilité",
                    height=220,
                )
                out_mask = gr.Image(
                    label="BLACK_SQUARE Masque prédit",
                    height=220,
                )

            out_stats = gr.Image(
                label="CHART_UP Graphiques d'analyse",
                height=230,
            )

            out_text = gr.Textbox(
                label="📋 Résumé",
                lines=10,
                interactive=False,
            )

    # ── Section exemples ─────────────────────────────────────────────────────
    gr.Markdown("---")
    gr.Markdown("### FRAME_WITH_PICTURE Exemples rapides — clique pour charger")

    # Charger 4 exemples depuis le dataset
    from pathlib import Path
    IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    example_paths = sorted([
        str(p) for p in Path(os.path.join(PROJECT_PATH, 'data', 'Images')).glob('*.*')
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])[:6]

    if example_paths:
        gr.Examples(
            examples=[[p, None, 0.5, True] for p in example_paths],
            inputs=[input_image, mask_input, threshold_slider, contour_check],
            outputs=[out_overlay, out_prob, out_mask, out_stats, out_text],
            fn=segment_image,
            cache_examples=False,
            label='Exemples du dataset ISIC'
        )

    # ── Aide ─────────────────────────────────────────────────────────────────
    with gr.Accordion("INFO Aide & Guide d'utilisation", open=False):
        gr.Markdown("""
        ### Comment utiliser l'interface ?

        1. **Upload une image** dermoscopique dans le cadre de gauche (jpg, png...)
        2. *(Optionnel)* Upload le **masque réel** correspondant pour obtenir les métriques
        3. **Ajuste le seuil** selon tes besoins :
           - `0.3 – 0.4` → Plus sensible : détecte plus de lésions (risque de fausses alarmes)
           - `0.5` → Équilibré (valeur recommandée)
           - `0.6 – 0.7` → Plus strict : moins de fausses alarmes (risque de manquer des lésions)
        4. Clique sur **MAGNIFYING_GLASS Segmenter**

        ### Interprétation des résultats
        | Sortie | Description |
        |--------|-------------|
        | PALETTE Overlay | Image originale avec la lésion surlignée en rouge |
        | THERMOMETER Carte probabilité | Pixels rouges = forte probabilité de lésion |
        | BLACK_SQUARE Masque prédit | Blanc = lésion détectée · Noir = peau saine |
        | CHART_UP Graphiques | Distribution des probabilités + métriques |

        ### Métriques (si masque réel fourni)
        - **Dice Score** > 0.85 → Excellente segmentation OK
        - **Recall** faible → Le modèle manque des lésions (baisse le seuil)
        - **Précision** faible → Trop de fausses alarmes (hausse le seuil)
        """)

    # ── Pied de page ─────────────────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; color:#888; font-size:0.85em; margin-top:12px;">
        ISIC Project · Rachid Bijigune · U-Net PyTorch · Mars 2026
    </div>
    """)

    # ── Connexion bouton → fonction ───────────────────────────────────────────
    btn.click(
        fn=segment_image,
        inputs=[input_image, mask_input, threshold_slider, contour_check],
        outputs=[out_overlay, out_prob, out_mask, out_stats, out_text]
    )

    # Mise à jour en temps réel quand le seuil change
    threshold_slider.release(
        fn=segment_image,
        inputs=[input_image, mask_input, threshold_slider, contour_check],
        outputs=[out_overlay, out_prob, out_mask, out_stats, out_text]
    )


# ── Lancement ─────────────────────────────────────────────────────────────────
print('LAUNCH Lancement de l\'interface...')
print('   → Un lien public sera généré ci-dessous')
print('   → Copie le lien et ouvre-le dans ton navigateur')

demo.launch(
    share=True,          # ← génère un lien public gradio.live
    debug=False,
    show_error=True,
    quiet=False,
)

---
## REFRESH Pour arrêter l'interface

In [ ]:
# Exécute cette cellule pour fermer l'interface
demo.close()
print('Interface fermée.')

---
## BULB Informations importantes

| Information | Détail |
|-------------|--------|
| **Lien public** | Valide **72 heures** après lancement |
| **Partage** | Tu peux envoyer le lien à n'importe qui |
| **Session** | Si Colab se déconnecte, relance les cellules depuis le début |
| **GPU** | Utilise automatiquement le GPU T4 pour les prédictions |
| **Formats** | JPG, PNG, BMP, TIFF acceptés |

### Pour utiliser l'interface :
1. Exécute toutes les cellules **dans l'ordre**
2. Attends le message `Running on public URL: https://...gradio.live`
3. **Copie** ce lien et ouvre-le dans ton navigateur
4. L'interface fonctionne depuis n'importe quel appareil (PC, téléphone...)